In [1]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

# Funzione per calcolare lo scarto percentuale
def scarto_percentuale(rad, shortwave_rad):
    return ((rad - shortwave_rad) / shortwave_rad) * 100

# Leggi il file CSV
file_path = 'ERA_PV_stazioni_clearsky.csv'  # Sostituisci con il percorso corretto
df = pd.read_csv(file_path)

# Assicurati che la colonna 'data' sia nel formato datetime
df['data'] = pd.to_datetime(df['data'], format='%Y-%m-%d')

# Aggrega i dati giornalmente, somma i valori di 'rad' e 'shortwave_radiation'
df_daily = df.groupby(['nome', 'data']).agg({'rad': 'sum', 'shortwave_radiation': 'sum'}).reset_index()

# Calcola lo scarto percentuale tra 'rad' e 'shortwave_radiation'
df_daily['scarto_percentuale'] = scarto_percentuale(df_daily['rad'], df_daily['shortwave_radiation'])
df_daily['valore_assoluto_scarto'] = df_daily['scarto_percentuale'].abs()

# Lista per memorizzare i risultati
risultati = []

# Cicla per ogni stazione
nomi_unici = df_daily['nome'].unique()

for nome in nomi_unici:
    # Filtra i dati per la stazione corrente
    df_nome = df_daily[df_daily['nome'] == nome]

    # Calcola la media dello scarto percentuale
    media_scarto_percentuale = df_nome['scarto_percentuale'].mean()

    # Calcola la media del valore assoluto dello scarto percentuale
    media_valore_assoluto_scarto = df_nome['valore_assoluto_scarto'].mean()

    # Usa l'indice del DataFrame come asse x per la regressione
    x = np.arange(len(df_nome))  # Array di indice

    # Calcola il coefficiente angolare della regressione lineare per scarto percentuale con segno
    slope_with_sign, intercept, r_value, p_value, std_err = linregress(x, df_nome['scarto_percentuale'])

    # Calcola il coefficiente angolare della regressione lineare per valore assoluto dello scarto percentuale
    slope_absolute, intercept, r_value, p_value, std_err = linregress(x, df_nome['valore_assoluto_scarto'])

    # Aggiungi i risultati alla lista
    risultati.append({
        'stazione': nome,
        'media_scarto_percentuale': media_scarto_percentuale,
        'media_valore_assoluto_scarto': media_valore_assoluto_scarto,
        'coefficiente_angolare_scarto': slope_with_sign,
        'coefficiente_angolare_assoluto': slope_absolute
    })

# Crea un DataFrame con i risultati
df_risultati = pd.DataFrame(risultati)

# Calcola la media totale di tutti gli scarti percentuali
media_totale_scarto_percentuale = df_daily['scarto_percentuale'].mean()

# Visualizza la tabella finale
print(df_risultati)
print(f"Media totale di tutti gli scarti percentuali: {media_totale_scarto_percentuale:.2f}%")


               stazione  media_scarto_percentuale  \
0                  ALBA                  2.164665   
1            ALESSANDRI                 -2.115933   
2                  ASTI                 -0.659205   
3             AVIGLIANA                 -1.541796   
4             BASALUZZO                 -3.589412   
5   BORGOFRANCO_D'IVREA                 -0.456022   
6           BORGOMANERO                -15.701054   
7                   BRA                 -0.852585   
8    BUTTIGLIERA_D'ASTI                 -1.500212   
9       CANDIA_CANAVESE                 -1.071472   
10           CARMAGNOLA                 -1.029098   
11    CASALE_MONFERRATO                 -0.542223   
12  COSTIGLIOLE_SALUZZO                 -0.517410   
13              CUMIANA                 -2.002689   
14                CUNEO                  2.565917   
15              FOSSANO                 -1.262782   
16   ISOLA_SANT'ANTONIO                 -4.838695   
17           MOMBALDONE                 -7.130